In [ ]:


# ══════════════════════════════════════════════════════
# 0. IMPORTS
# ══════════════════════════════════════════════════════
import os, copy, time, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
import seaborn as sns
from PIL import Image
from collections import defaultdict
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# ── Added: logistic regression for single‑feature modelling ────────
from sklearn.linear_model import LogisticRegression

try:
    import timm; HAS_TIMM = True
except ImportError:
    HAS_TIMM = False; print("Tip: pip install timm to unlock ConvNeXt/Swin/RegNet")

from sklearn.metrics import (
    confusion_matrix, roc_curve, auc, precision_recall_curve,
    average_precision_score, accuracy_score
)
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
from scipy import stats

try:
    from skimage.metrics import structural_similarity as ssim; HAS_SSIM = True
except ImportError:
    HAS_SSIM = False


# ══════════════════════════════════════════════════════
# 1. CONFIGURATION
# ══════════════════════════════════════════════════════
CONFIG = {
    'result_dir':   './results',
    'out_dir':      './results/multi_model',
    'weights_dir':  './results/weights',
    # ── Added: path to the CSV file with predicted probabilities ──
    'scores_csv':   './results/multi_model/prediction_scores_all.csv',
    'img_size':     224,
    'batch_size':   8,
    'num_classes':  2,
    'class_names':  ['Insensitive', 'Sensitive'],
    'num_workers':  0,
    'epochs':       10,
    'lr':           1e-4,
    'weight_decay': 1e-2,
    'seed':         42,
    'bootstrap_n':  1000,
    'dpi':          300,
    'device':       'cuda' if torch.cuda.is_available() else 'cpu',
}

for d in [CONFIG['out_dir'], CONFIG['weights_dir']]:
    os.makedirs(d, exist_ok=True)

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])


# ══════════════════════════════════════════════════════
# 2. BIORENDER STYLE
# ══════════════════════════════════════════════════════
def setup_style():
    """
    Configure matplotlib to mimic the clean, high‑end academic plotting
    style of BioRender.
    """
    plt.rcParams.update({
        'font.family':       'sans-serif',
        'font.sans-serif':   ['Arial', 'Helvetica', 'DejaVu Sans', 'Liberation Sans'],
        'font.size':         11,
        'axes.facecolor':    '#FAFBFC',         # very light blue‑grey background
        'figure.facecolor':  'white',           # full white canvas
        'axes.edgecolor':    '#2C3E50',         # dark navy border
        'axes.linewidth':    1.5,
        'axes.grid':         True,
        'grid.color':        '#E2E8F0',         # BioRender‑style grid colour
        'grid.linestyle':    '--',
        'grid.linewidth':    0.8,
        'grid.alpha':        0.6,
        'axes.axisbelow':    True,
        'xtick.color':       '#2C3E50',
        'ytick.color':       '#2C3E50',
        'xtick.direction':   'out',
        'ytick.direction':   'out',
        'text.color':        '#2C3E50',
        'axes.labelcolor':   '#2C3E50',
        'axes.titlesize':    14,
        'axes.titleweight':  'bold',
        'axes.labelsize':    12,
        'axes.labelweight':  'bold',
        'legend.frameon':    True,
        'legend.framealpha': 0.95,
        'legend.facecolor':  'white',
        'legend.edgecolor':  '#E2E8F0',
        'legend.fontsize':   9,
        'lines.linewidth':   2.0,
        'lines.markersize':  6,
        'pdf.fonttype':      42,
        'ps.fonttype':       42,
    })

setup_style()

# Advanced Morandi / biomedical colour palette (warm and cool hues often seen in BioRender)
PALETTE = [
    '#E07A5F', # Coral Red
    '#3D5A80', # Deep Navy
    '#81B29A', # Sage Green
    '#F2CC8F', # Cream Yellow
    '#9D8189', # Lilac Purple
    '#E5989B', # Rose Pink
    '#457B9D', # Steel Blue
    '#2A9D8F', # Teal
    '#E9C46A', # Mustard Yellow
    '#F4A261', # Apricot Orange
    '#264653', # Dark Slate
    '#E76F51', # Terracotta
    '#A8DADC', # Light Blue
    '#6D6875', # Taupe
    '#B5838D', # Dusty Rose
    '#55A630', # Grass Green
    '#D90429', # Brick Red
    '#0077B6', # Bright Blue
    '#C1121F', # Crimson
    '#03045E', # Midnight Blue
]

CMAP_AUC  = plt.cm.RdYlGn
CMAP_BLUE = LinearSegmentedColormap.from_list(
    'biorender_blue', ['#F0F4F8', '#BDD7E7', '#6BAED6', '#2171B5', '#08306B'])
CMAP_HEAT = LinearSegmentedColormap.from_list(
    'biorender_heat', ['#FFFFFF', '#FEE0D2', '#FC9272', '#DE2D26'])


# ══════════════════════════════════════════════════════
# 3. I/O HELPERS
# ══════════════════════════════════════════════════════
def out_path(name): return os.path.join(CONFIG['out_dir'], name)

def save_fig(fig, stem):
    fig.savefig(out_path(stem + '.pdf'), format='pdf', bbox_inches='tight', facecolor='white')
    fig.savefig(out_path(stem + '.png'), dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  [fig] {stem}.pdf")

def save_csv(df, stem):
    path = out_path(stem + '.csv')
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"  [csv] {stem}.csv")


# ══════════════════════════════════════════════════════
# 4. DATASET  (kept for reference; not called in run() when using CSV)
# ══════════════════════════════════════════════════════
class MRIDataset(Dataset):
    def __init__(self, df, transform=None):
        self.data = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, int(row['label']), str(row['patient_id']), row['path']


def get_transforms(train=True):
    """
    Pre‑processing identical to the paper:
      - extract 2D slice with maximum tumour area (handled externally)
      - normalise to 0–255
      - resize to 224×224 with bilinear interpolation
      - convert grayscale to RGB via channel replication
      - apply standard ImageNet normalisation
    """
    base = [
        transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]
    if train:
        aug = [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        ]
        return transforms.Compose([base[0]] + aug + base[1:])
    return transforms.Compose(base)


def load_data():
    train_csv = os.path.join(CONFIG['result_dir'], 'train_dataset.csv')
    test_csv  = os.path.join(CONFIG['result_dir'], 'test_dataset.csv')

    if os.path.exists(train_csv) and os.path.exists(test_csv):
        print("Loading existing train/test split...")
        return pd.read_csv(train_csv), pd.read_csv(test_csv)

    print("Splitting dataset.csv into 80% Train, 20% Test...")
    df = pd.read_csv('dataset.csv')
    train_df, test_df = train_test_split(df, test_size=0.2,
                                         random_state=CONFIG['seed'],
                                         stratify=df['label'])
    train_df.to_csv(train_csv, index=False, encoding='utf-8-sig')
    test_df.to_csv(test_csv,   index=False, encoding='utf-8-sig')
    return train_df, test_df


# ══════════════════════════════════════════════════════
# 5. MODEL REGISTRY  (kept for reference; not called when using CSV)
# ══════════════════════════════════════════════════════
class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, ch//r, bias=False), nn.ReLU(inplace=True),
            nn.Linear(ch//r, ch, bias=False), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), x.size(1), 1, 1)


class SEResNet50(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        b = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
        self.layer0 = nn.Sequential(b.conv1, b.bn1, b.relu, b.maxpool)
        self.layer1, self.layer2 = b.layer1, b.layer2
        self.layer3, self.layer4 = b.layer3, b.layer4
        self.se = SEBlock(2048)
        self.avgpool = b.avgpool
        self.fc = nn.Linear(2048, num_classes)
    def forward(self, x):
        for layer in [self.layer0, self.layer1, self.layer2, self.layer3, self.layer4]:
            x = layer(x)
        x = self.se(x)
        return self.fc(torch.flatten(self.avgpool(x), 1))


def build_registry(num_classes):
    """Returns {name: {model, target_layer, paper, params_m}}."""
    R = {}
    def reg(name, model, tl, paper, params_m):
        R[name] = dict(model=model, target_layer=tl, paper=paper, params_m=params_m)

    m = models.alexnet(weights='IMAGENET1K_V1')
    m.classifier[-1] = nn.Sequential(nn.Dropout(0.5), nn.Linear(4096, num_classes))
    reg('AlexNet', m, m.features[-3], 'Krizhevsky et al., NeurIPS 2012', 57.0)

    for arch, fn, params in [('VGG16', models.vgg16, 138.4), ('VGG19', models.vgg19, 143.7)]:
        m = fn(weights='IMAGENET1K_V1')
        m.classifier[-1] = nn.Sequential(nn.Dropout(0.5), nn.Linear(4096, num_classes))
        reg(arch, m, m.features[-1], 'Simonyan & Zisserman, ICLR 2015', params)

    for arch, fn, fc_in, params in [
        ('ResNet18',  models.resnet18,  512,  11.7),
        ('ResNet34',  models.resnet34,  512,  21.8),
        ('ResNet50',  models.resnet50,  2048, 25.6),
        ('ResNet101', models.resnet101, 2048, 44.5),
    ]:
        m = fn(weights='IMAGENET1K_V1')
        mid = fc_in // 2
        m.fc = nn.Sequential(
            nn.Dropout(0.6), nn.Linear(fc_in, mid),
            nn.ReLU(inplace=True), nn.Dropout(0.6), nn.Linear(mid, num_classes))
        reg(arch, m, m.layer4[-1], 'He et al., CVPR 2016', params)

    for arch, fn, fc_in, params in [
        ('DenseNet121', models.densenet121, 1024, 8.0),
        ('DenseNet169', models.densenet169, 1664, 14.1),
        ('DenseNet201', models.densenet201, 1920, 20.0),
    ]:
        m = fn(weights='IMAGENET1K_V1')
        mid = fc_in // 2
        m.classifier = nn.Sequential(
            nn.Dropout(0.6), nn.Linear(fc_in, mid),
            nn.ReLU(inplace=True), nn.Dropout(0.6), nn.Linear(mid, num_classes))
        reg(arch, m, m.features.denseblock4, 'Huang et al., CVPR 2017', params)

    m = models.mobilenet_v2(weights='IMAGENET1K_V1')
    m.classifier[-1] = nn.Sequential(nn.Dropout(0.5), nn.Linear(1280, num_classes))
    reg('MobileNetV2', m, m.features[-1], 'Sandler et al., CVPR 2018', 3.4)

    m = models.mobilenet_v3_large(weights='IMAGENET1K_V1')
    m.classifier[-1] = nn.Sequential(nn.Dropout(0.5), nn.Linear(1280, num_classes))
    reg('MobileNetV3L', m, m.features[-1], 'Howard et al., ICCV 2019', 5.5)

    m = models.squeezenet1_1(weights='IMAGENET1K_V1')
    m.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=1)
    m.num_classes = num_classes
    reg('SqueezeNet', m, m.features[-1], 'Iandola et al., arXiv 2016', 1.2)

    m = models.shufflenet_v2_x1_0(weights='IMAGENET1K_V1')
    m.fc = nn.Sequential(nn.Dropout(0.7), nn.Linear(1024, num_classes))
    reg('ShuffleNetV2', m, m.conv5, 'Ma et al., ECCV 2018', 2.3)

    m = SEResNet50(num_classes=num_classes, pretrained=True)
    m.fc = nn.Sequential(nn.Dropout(0.8), nn.Linear(2048, num_classes))
    reg('SE-ResNet50', m, m.layer4[-1], 'Hu et al., CVPR 2018', 28.1)

    m = models.efficientnet_b0(weights='IMAGENET1K_V1')
    m.classifier[-1] = nn.Sequential(nn.Dropout(0.5), nn.Linear(1280, num_classes))
    reg('EfficientNet-B0', m, m.features[-1], 'Tan & Le, ICML 2019', 5.3)

    m = models.efficientnet_b4(weights='IMAGENET1K_V1')
    m.classifier[-1] = nn.Sequential(nn.Dropout(0.7), nn.Linear(1792, num_classes))
    reg('EfficientNet-B4', m, m.features[-1], 'Tan & Le, ICML 2019', 19.3)

    if HAS_TIMM:
        for name, model_name, tl_fn, paper, params in [
            ('RegNetY-400MF', 'regnety_004',
             lambda m: m.s4.b1.conv3 if hasattr(m, 's4') else list(m.modules())[-5],
             'Radosavovic et al., CVPR 2020', 4.3),
            ('ConvNeXt-Tiny', 'convnext_tiny',
             lambda m: m.stages[-1].blocks[-1].conv_dw,
             'Liu et al., CVPR 2022', 28.6),
            ('Swin-Tiny', 'swin_tiny_patch4_window7_224',
             lambda m: m.layers[-1].blocks[-1].norm2,
             'Liu et al., ICCV 2021', 28.3),
        ]:
            try:
                m = timm.create_model(model_name, pretrained=True,
                                      num_classes=num_classes, drop_rate=0.5)
                reg(name, m, tl_fn(m), paper, params)
            except Exception as e:
                print(f"  Warning: {name} failed to load — {e}")

    return R


# ══════════════════════════════════════════════════════
# 6. GRAD-CAM++  (kept for reference; not called when using CSV)
# ══════════════════════════════════════════════════════
class UniversalGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = self.gradients = None
        self._disable_inplace(model)
        self._register()

    def _disable_inplace(self, module):
        for child in module.modules():
            if hasattr(child, 'inplace'): child.inplace = False

    def _register(self):
        self.target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o))
        self.target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0]))

    def _to_spatial(self, t):
        if t is None: return None
        if t.ndim == 3:
            B, L, C = t.shape; H = W = int(L**0.5)
            t = t.permute(0, 2, 1).reshape(B, C, H, W)
        return t if t.ndim == 4 else None

    def __call__(self, x, class_idx=None, method='gradcam++'):
        self.model.eval()
        self.activations = self.gradients = None
        x = x.to(CONFIG['device']).requires_grad_(True)
        out = self.model(x)
        if class_idx is None: class_idx = out.argmax(1).item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(out); one_hot[0, class_idx] = 1.
        out.backward(gradient=one_hot, retain_graph=True)

        acts  = self._to_spatial(self.activations)
        grads = self._to_spatial(self.gradients)
        if acts is None or grads is None: return None, class_idx

        if method == 'gradcam++':
            alpha = grads**2 / (2*grads**2 + (grads**3)*acts.sum((2,3), keepdim=True) + 1e-8)
            weights = (alpha * F.relu(grads)).mean((2, 3), keepdim=True)
        else:
            weights = grads.mean((2, 3), keepdim=True)

        cam = F.relu((weights * acts).sum(1, keepdim=True))[0, 0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

    def overlay(self, cam, raw_rgb, alpha=0.45):
        h, w = raw_rgb.shape[:2]
        cam_up = cv2.resize(cam, (w, h))
        hm = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cam_up), cv2.COLORMAP_JET),
                          cv2.COLOR_BGR2RGB)
        return (raw_rgb * (1 - alpha) + hm * alpha).astype(np.uint8), hm


# ══════════════════════════════════════════════════════
# 7. TRAINING  (kept for reference; not called when using CSV)
# ══════════════════════════════════════════════════════
MIXUP_MODELS = {
    'ResNet18', 'ResNet34', 'ResNet50', 'ResNet101',
    'DenseNet121', 'DenseNet169', 'DenseNet201'
}


def train_model(name, model, train_loader, use_mixup=False):
    from sklearn.metrics import roc_auc_score
    device = CONFIG['device']
    model  = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'],
                            weight_decay=CONFIG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
    criterion = nn.CrossEntropyLoss(label_smoothing=0.25)

    weight_path = os.path.join(CONFIG['weights_dir'], f'{name}.pth')
    best_auc, history = 0., defaultdict(list)

    if use_mixup:
        print(f"  [{name}] Mixup enabled (alpha=0.4)")

    for epoch in range(CONFIG['epochs']):
        model.train()
        run_loss = correct = total = 0
        t_probs, t_lbls = [], []

        for imgs, labels, _, _ in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()

            if use_mixup:
                lam = np.random.beta(0.4, 0.4)
                idx = torch.randperm(imgs.size(0)).to(device)
                mixed_imgs = lam * imgs + (1 - lam) * imgs[idx]
                out  = model(mixed_imgs)
                loss = lam * criterion(out, labels) + (1 - lam) * criterion(out, labels[idx])
                correct += (out.argmax(1) == labels).sum().item()
            else:
                out  = model(imgs)
                loss = criterion(out, labels)
                correct += (out.argmax(1) == labels).sum().item()

            loss.backward()
            optimizer.step()
            run_loss += loss.item()
            total    += labels.size(0)

            with torch.no_grad():
                t_probs += torch.softmax(out, 1)[:, 1].cpu().tolist()
                t_lbls  += labels.cpu().tolist()

        train_acc = correct / total
        train_auc = roc_auc_score(t_lbls, t_probs) if len(set(t_lbls)) > 1 else 0.5

        history['train_loss'].append(run_loss / len(train_loader))
        history['train_acc'].append(train_acc)
        history['train_auc'].append(train_auc)
        scheduler.step()

        if train_auc >= best_auc:
            best_auc = train_auc
            torch.save(model.state_dict(), weight_path)

        print(f"  [{name}] Ep{epoch+1:02d} "
              f"Loss={history['train_loss'][-1]:.4f} "
              f"TrainAcc={train_acc:.3f} "
              f"TrainAUC={train_auc:.3f}"
              + (' ✓' if train_auc >= best_auc else ''))

    return dict(history), weight_path


# ══════════════════════════════════════════════════════
# 8. EVALUATION (original deep‑learning version, kept for reference)
# ══════════════════════════════════════════════════════
def _metrics(y, p, pd_):
    if len(set(y)) < 2:
        return dict(acc=0, sen=0, spe=0, pre=0, f1=0, auc=0.5, ap=0,
                    OR=1, OR_lo=0.1, OR_hi=10)
    cm = confusion_matrix(y, pd_)
    tn, fp, fn, tp = (cm.ravel() if cm.size == 4 else [0, 0, 0, 0])
    acc = (tp+tn) / (tp+tn+fp+fn+1e-8)
    sen = tp / (tp+fn+1e-8);  spe = tn / (tn+fp+1e-8)
    pre = tp / (tp+fp+1e-8);  f1  = 2*pre*sen / (pre+sen+1e-8)
    fpr_, tpr_, _ = roc_curve(y, p);  roc_auc = auc(fpr_, tpr_)
    ap = average_precision_score(y, p)
    tp_, fp_, fn_, tn_ = max(tp, 0.5), max(fp, 0.5), max(fn, 0.5), max(tn, 0.5)
    OR = (tp_*tn_) / (fp_*fn_)
    se_log = np.sqrt(1/tp_ + 1/fp_ + 1/fn_ + 1/tn_)
    OR_lo = np.exp(np.log(OR) - 1.96*se_log)
    OR_hi = np.exp(np.log(OR) + 1.96*se_log)
    return dict(acc=acc, sen=sen, spe=spe, pre=pre, f1=f1, auc=roc_auc, ap=ap,
                OR=OR, OR_lo=OR_lo, OR_hi=OR_hi)


def evaluate_model(name, model, loader, weight_path):
    """Original evaluation (kept for reference)."""
    device = CONFIG['device']
    model  = model.to(device)
    if os.path.exists(weight_path):
        model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
    model.eval()

    y_all, p_all, pd_all, pid_all = [], [], [], []
    with torch.no_grad():
        for imgs, labels, pids, _ in loader:
            out     = model(imgs.to(device))
            p_all  += torch.softmax(out, 1)[:, 1].cpu().tolist()
            pd_all += out.argmax(1).cpu().tolist()
            y_all  += labels.tolist()
            pid_all += list(pids)

    y = np.array(y_all); p = np.array(p_all); pd_ = np.array(pd_all)
    metrics = _metrics(y, p, pd_)

    ci = defaultdict(list)
    for _ in range(CONFIG['bootstrap_n']):
        idx = np.random.choice(len(y), len(y), replace=True)
        if len(set(y[idx])) < 2: continue
        m_b = _metrics(y[idx], p[idx], pd_[idx])
        for k, v in m_b.items(): ci[k].append(v)
    ci = {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in ci.items()}

    fpr_, tpr_, _ = roc_curve(y, p)
    pre_, rec_, _ = precision_recall_curve(y, p)
    return dict(metrics=metrics, ci=ci,
                labels=y, probs=p, preds=pd_, pids=pid_all,
                fpr=fpr_, tpr=tpr_, precision=pre_, recall=rec_)


# ══════════════════════════════════════════════════════
# 8-NEW. LOGISTIC‑REGRESSION EVALUATION USING PREDICTED SCORES
#    Input: prob_sensitive column → logistic regression → predict true_label
#    Output format identical to evaluate_model() for seamless reuse.
# ══════════════════════════════════════════════════════
def _build_result_dict(y: np.ndarray, p: np.ndarray,
                       pd_: np.ndarray, pids: list) -> dict:
    metrics = _metrics(y, p, pd_)

    ci = defaultdict(list)
    for _ in range(CONFIG['bootstrap_n']):
        idx = np.random.choice(len(y), len(y), replace=True)
        if len(set(y[idx])) < 2: continue
        m_b = _metrics(y[idx], p[idx], pd_[idx])
        for k, v in m_b.items(): ci[k].append(v)
    ci = {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in ci.items()}

    fpr_, tpr_, _ = roc_curve(y, p)
    pre_, rec_, _ = precision_recall_curve(y, p)
    return dict(metrics=metrics, ci=ci,
                labels=y, probs=p, preds=pd_, pids=pids,
                fpr=fpr_, tpr=tpr_, precision=pre_, recall=rec_)


def _evaluate_lr(model_name: str, scores_df: pd.DataFrame) -> tuple:
    mdf       = scores_df[scores_df['model'] == model_name].copy()
    train_sub = mdf[mdf['split'] == 'Train'].reset_index(drop=True)
    test_sub  = mdf[mdf['split'] == 'Test'].reset_index(drop=True)

    X_tr = train_sub[['prob_sensitive']].values
    y_tr = train_sub['true_label'].values.astype(int)
    X_te = test_sub[['prob_sensitive']].values
    y_te = test_sub['true_label'].values.astype(int)

    pids_tr = train_sub['patient_id'].astype(str).tolist()
    pids_te = test_sub['patient_id'].astype(str).tolist()

    lr = LogisticRegression(random_state=CONFIG['seed'], max_iter=1000,
                            class_weight='balanced')
    lr.fit(X_tr, y_tr)

    p_tr  = lr.predict_proba(X_tr)[:, 1]
    pd_tr = lr.predict(X_tr)
    p_te  = lr.predict_proba(X_te)[:, 1]
    pd_te = lr.predict(X_te)

    train_res = _build_result_dict(y_tr, p_tr, pd_tr, pids_tr)
    test_res  = _build_result_dict(y_te, p_te, pd_te, pids_te)
    return train_res, test_res




# ══════════════════════════════════════════════════════
# 21. DeLong TEST HEATMAP (BioRender style)
# ══════════════════════════════════════════════════════
def _delong(y, p1, p2):
    def _stats(y, p):
        pos = p[y == 1]; neg = p[y == 0]
        n_p, n_n = len(pos), len(neg)
        if not (n_p and n_n): return 0, 0
        v10 = np.array([np.mean(pi > neg) for pi in pos])
        v01 = np.array([np.mean(pos > ni) for ni in neg])
        return np.mean(v10 > 0.5), np.var(v10, ddof=1)/n_p + np.var(v01, ddof=1)/n_n
    a1, s1 = _stats(y, p1); a2, s2 = _stats(y, p2)
    denom = np.sqrt(s1 + s2 + 1e-12)
    z = (a1 - a2) / denom if denom > 1e-10 else 0.
    return z, 2*(1 - stats.norm.cdf(abs(z)))


def plot_delong(train_res, test_res):
    records = []
    for results, split in [(train_res, 'Train'), (test_res, 'Test')]:
        names = list(results.keys()); n = len(names)
        p_mat = np.ones((n, n)); z_mat = np.zeros((n, n))
        y_ref = results[names[0]]['labels']
        for i in range(n):
            for j in range(i+1, n):
                try:
                    z, p = _delong(y_ref, results[names[i]]['probs'],
                                   results[names[j]]['probs'])
                except: z, p = 0, 1.
                p_mat[i, j] = p_mat[j, i] = p
                z_mat[i, j] = z; z_mat[j, i] = -z
                records.append(dict(split=split, model_A=names[i], model_B=names[j],
                                    z=f"{z:.4f}", p=f"{p:.4f}"))

        fig, axes = plt.subplots(1, 2, figsize=(max(18, n*1.5), max(8, n*1.0)))
        fig.suptitle(f'DeLong AUC Significance Test — {split} Set',
                     fontsize=16, fontweight='bold', color='#1B3A5C', y=1.02)

        sns.heatmap(-np.log10(p_mat+1e-10), annot=p_mat, fmt='.3f',
                    xticklabels=names, yticklabels=names, cmap=plt.cm.YlGnBu,
                    ax=axes[0], annot_kws={'fontsize': 9}, linewidths=0.5, linecolor='white')
        axes[0].set_title('-log10(p-value)  [Dark Blue = Highly Significant]', fontsize=12, fontweight='bold', pad=10)
        axes[0].tick_params(labelsize=9)

        vmax = np.abs(z_mat).max() + 0.1
        sns.heatmap(z_mat, annot=True, fmt='.2f',
                    xticklabels=names, yticklabels=names,
                    cmap=plt.cm.RdBu_r, vmin=-vmax, vmax=vmax,
                    ax=axes[1], annot_kws={'fontsize': 9}, linewidths=0.5, linecolor='white')
        axes[1].set_title('Z-statistic (Row vs Column)', fontsize=12, fontweight='bold', pad=10)
        axes[1].tick_params(labelsize=9)

        plt.tight_layout(w_pad=4)
        save_fig(fig, f"13_delong_{split.lower()}")
    save_csv(pd.DataFrame(records), "13_delong")


# ══════════════════════════════════════════════════════
# 22. FOREST PLOT (BioRender style)
# ══════════════════════════════════════════════════════
def plot_forest(train_res, test_res):
    def _create_single_forest(res_dict, split_name, color_theme, marker_style):
        names = list(res_dict.keys())
        names = sorted(names, key=lambda n: res_dict[n]['metrics']['OR'], reverse=True)
        n     = len(names)
        aucs  = [res_dict[nm]['metrics']['auc']   for nm in names]
        ORs   = [res_dict[nm]['metrics']['OR']    for nm in names]
        los   = [res_dict[nm]['metrics']['OR_lo'] for nm in names]
        his   = [res_dict[nm]['metrics']['OR_hi'] for nm in names]

        fig = plt.figure(figsize=(14, max(7, n*0.7+2)))
        gs  = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.2, 3], wspace=0.1)

        # Left: AUC display
        ax_left = fig.add_subplot(gs[0])
        norm    = Normalize(vmin=0.5, vmax=1.0)
        cmap_auc = CMAP_AUC
        y_pos   = np.arange(n)[::-1]
        box_w, box_h = 0.5, 0.6; col_x = 0.5

        for row_i, (name, val) in enumerate(zip(names, aucs)):
            yp    = y_pos[row_i]
            color = cmap_auc(norm(val))
            rect  = mpatches.FancyBboxPatch((col_x - box_w/2, yp - box_h/2), box_w, box_h,
                                            boxstyle="round,pad=0.05",
                                            facecolor=color, edgecolor='white', lw=1.5)
            ax_left.add_patch(rect)
            ax_left.text(col_x, yp, f'{val:.3f}', ha='center', va='center',
                         fontsize=10, fontweight='bold',
                         color='white' if val < 0.75 else '#1B3A5C')

        ax_left.set_xlim(0, 1); ax_left.set_ylim(-0.8, n-0.2)
        ax_left.set_yticks(y_pos); ax_left.set_yticklabels(names, fontsize=11, fontweight='bold', color='#2C3E50')
        ax_left.set_xticks([col_x]); ax_left.set_xticklabels(['AUC'], fontsize=12, fontweight='bold')
        ax_left.xaxis.set_ticks_position('top')
        ax_left.set_title(f'{split_name} Efficacy', fontsize=13, fontweight='bold', color='#1B3A5C', pad=25)
        ax_left.grid(False)
        for sp in ax_left.spines.values(): sp.set_visible(False)
        ax_left.axhline(y=-0.5, color='#E2E8F0', lw=1.5)

        # Right: forest plot
        ax_right = fig.add_subplot(gs[1])

        # Alternating row background for readability
        for i in range(n):
            if i % 2 == 0:
                ax_right.axhspan(y_pos[i]-0.5, y_pos[i]+0.5, color='#F8FAFC', zorder=0)

        ax_right.axvline(1.0, color='#94A3B8', lw=1.5, ls='--', zorder=1)

        for row_i, name in enumerate(names):
            yp     = y_pos[row_i]
            or_val = ORs[row_i]; lo_val = los[row_i]; hi_val = his[row_i]

            ax_right.errorbar(or_val, yp,
                              xerr=[[max(0, or_val - lo_val)], [hi_val - or_val]],
                              fmt=marker_style, color=color_theme, ms=9, lw=2.5,
                              capsize=6, capthick=2, zorder=3, markeredgecolor='white', markeredgewidth=1)

            ax_right.text(hi_val * 1.15, yp,
                          f'{or_val:.2f} [{lo_val:.2f} - {hi_val:.2f}]',
                          va='center', fontsize=10, color='#475569', fontweight='bold')

        ax_right.set_xscale('log')
        ax_right.set_xlabel('Odds Ratio (OR) — Log Scale', fontsize=12, fontweight='bold', labelpad=10)
        ax_right.set_title(f'{split_name} Odds Ratio Forest Plot', fontsize=13, fontweight='bold', color='#1B3A5C', pad=20)
        ax_right.set_ylim(-0.8, n-0.2)
        ax_right.set_yticks(y_pos); ax_right.set_yticklabels([])

        ax_right.spines['top'].set_visible(False)
        ax_right.spines['right'].set_visible(False)
        ax_right.spines['left'].set_visible(False)

        fig.suptitle(f'Predictive Model Performance Summary — {split_name} Set',
                     fontsize=18, fontweight='bold', color='#1B3A5C', y=1.03)
        plt.tight_layout()
        stem = f"14_forest_plot_{split_name.lower()}"
        save_fig(fig, stem)
        records = [{'Model': nm, 'AUC': f"{a:.4f}", 'OR': f"{o:.4f}",
                    'CI_lo': f"{l:.4f}", 'CI_hi': f"{h:.4f}"}
                   for nm, a, o, l, h in zip(names, aucs, ORs, los, his)]
        save_csv(pd.DataFrame(records), stem)

    _create_single_forest(train_res, 'Train', color_theme='#E07A5F', marker_style='D')
    _create_single_forest(test_res,  'Test',  color_theme='#2A9D8F', marker_style='o')


# ══════════════════════════════════════════════════════
# 23. METRICS SUMMARY TABLE (BioRender style)
# ══════════════════════════════════════════════════════
def export_summary(train_res, test_res, registry):
    mkeys = ['acc', 'sen', 'spe', 'pre', 'f1', 'auc', 'ap', 'OR']
    rows  = []
    for name in test_res:
        row = dict(Model=name,
                   Paper=registry.get(name, {}).get('paper', '-'),
                   Params_M=registry.get(name, {}).get('params_m', '-'))
        for split, res in [('Train', train_res), ('Test', test_res)]:
            if name not in res: continue
            for mk in mkeys:
                v  = res[name]['metrics'].get(mk, 0)
                ci = res[name]['ci'].get(mk, (0, 0))
                row[f'{split}_{mk.upper()}'] = f"{v:.4f}"
                if mk != 'OR':
                    row[f'{split}_{mk.upper()}_CI'] = f"[{ci[0]:.4f},{ci[1]:.4f}]"
        rows.append(row)

    df = pd.DataFrame(rows).sort_values('Test_AUC', ascending=False)
    save_csv(df, "15_metrics_summary")

    disp_cols = (['Model', 'Params_M'] +
                 [f'{s}_{m.upper()}' for s in ['Train', 'Test']
                  for m in ['acc', 'sen', 'spe', 'pre', 'f1', 'auc', 'ap']])
    df_d = df[[c for c in disp_cols if c in df.columns]]
    fig, ax = plt.subplots(figsize=(max(24, len(df_d.columns)*1.8), len(df_d)*0.7+2))
    ax.axis('off')

    tbl = ax.table(cellText=df_d.values, colLabels=df_d.columns,
                   loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 2.2)

    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor('#1B3A5C')
            cell.set_text_props(color='white', fontweight='bold', fontsize=10)
        else:
            cell.set_facecolor('#F8FAFC' if row % 2 == 0 else 'white')
            cell.set_text_props(color='#2C3E50')
        cell.set_edgecolor('#E2E8F0')
        cell.set_linewidth(1.5)

    fig.suptitle('Multi-Model Performance Summary (Sorted by Test AUC)',
                 fontsize=16, fontweight='bold', color='#1B3A5C', y=0.98)
    plt.tight_layout()
    save_fig(fig, "15_metrics_summary_table")


# ══════════════════════════════════════════════════════
# Additional: export per‑sample prediction scores (train + test)
# ══════════════════════════════════════════════════════
def export_prediction_scores(train_results, test_results):
    all_records = []
    for results, split in [(train_results, 'Train'), (test_results, 'Test')]:
        split_records = []
        for name, res in results.items():
            y    = res['labels']
            p    = res['probs']
            pd_  = res['preds']
            pids = res['pids']
            for pid, true, pred, prob in zip(pids, y, pd_, p):
                record = dict(
                    model            = name,
                    patient_id       = pid,
                    true_label       = int(true),
                    true_label_name  = CONFIG['class_names'][int(true)],
                    pred_label       = int(pred),
                    pred_label_name  = CONFIG['class_names'][int(pred)],
                    prob_sensitive   = round(float(prob), 6),
                    prob_insensitive = round(1 - float(prob), 6),
                    correct          = int(true) == int(pred),
                    split            = split,
                )
                split_records.append(record)
                all_records.append(record)

        df_split = pd.DataFrame(split_records)
        df_split = df_split.sort_values(['model', 'patient_id']).reset_index(drop=True)
        save_csv(df_split, f"16_prediction_scores_{split.lower()}")

        pivot = df_split.pivot_table(
            index   = ['patient_id', 'true_label', 'true_label_name'],
            columns = 'model',
            values  = 'prob_sensitive',
            aggfunc = 'first'
        ).reset_index()
        pivot.columns.name = None
        save_csv(pivot, f"16_prediction_scores_{split.lower()}_wide")

        print(f"  [{split}] {len(df_split)} records exported "
              f"({df_split['correct'].sum()} correct / {len(df_split)} total predictions)")

    df_all = pd.DataFrame(all_records).sort_values(
        ['split', 'model', 'patient_id']).reset_index(drop=True)
    save_csv(df_all, "16_prediction_scores_all")
    print(f"  [All]  {len(df_all)} total records saved → 16_prediction_scores_all.csv")


# ══════════════════════════════════════════════════════
# 24. MAIN PIPELINE
# ══════════════════════════════════════════════════════
def run():
    print(f"\n{'='*65}")
    print("  Multi-Model LR Framework (BioRender Styled) — Starting")
    print(f"  Output: {CONFIG['out_dir']}")
    print(f"{'='*65}\n")

    # ── Load the prediction‑score CSV ─────────────────────────────────
    scores_path = CONFIG['scores_csv']
    if not os.path.exists(scores_path):
        raise FileNotFoundError(
            f"Prediction scores CSV not found: {scores_path}\n"
            "Please run the deep‑learning pipeline first to generate "
            "16_prediction_scores_all.csv, then re‑run this script."
        )

    scores_df = pd.read_csv(scores_path, encoding='utf-8-sig')
    print(f"  Loaded: {scores_path}  shape={scores_df.shape}")

    required_cols = {'model', 'patient_id', 'true_label', 'prob_sensitive', 'split'}
    missing = required_cols - set(scores_df.columns)
    if missing:
        raise ValueError(f"Missing columns in scores CSV: {missing}")

    model_names = scores_df['model'].unique().tolist()
    print(f"  Models found ({len(model_names)}): {model_names}\n")

    train_results = {}
    test_results  = {}
    histories     = {}

    for name in model_names:
        print(f"\n{'─'*55}\n  ▶  {name}\n{'─'*55}")

        tr_res, te_res = _evaluate_lr(name, scores_df)
        train_results[name] = tr_res
        test_results[name]  = te_res
        histories[name]     = {}

        m_tr = tr_res['metrics']; m_te = te_res['metrics']
        print(f"  [Train] AUC={m_tr['auc']:.4f}  F1={m_tr['f1']:.4f}  OR={m_tr['OR']:.3f}")
        print(f"  [Test]  AUC={m_te['auc']:.4f}  F1={m_te['f1']:.4f}  OR={m_te['OR']:.3f}")

    print(f"\n{'='*55}\n  Generating BioRender‑style visualisations...\n{'='*55}")

    plot_roc(train_results, test_results)
    plot_pr(train_results, test_results)
    plot_radar(train_results, test_results)
    plot_metric_bars(train_results, test_results)
    plot_cm_grid(train_results, test_results)
    plot_score_distribution(train_results, test_results)
    plot_calibration(train_results, test_results)
    plot_complexity(train_results, test_results, {})
    plot_delong(train_results, test_results)
    plot_forest(train_results, test_results)
    export_summary(train_results, test_results, {})
    export_prediction_scores(train_results, test_results)




if __name__ == '__main__':
    run()